<a href="https://colab.research.google.com/github/Tejay7861/Phishing-Detection-/blob/main/smooth_dataset_with_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# 1. Import Libraries

import pandas as pd
import numpy as np
from google.colab import files
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder


# 2. Upload Dataset

uploaded = files.upload()

# load dataset
df = pd.read_csv('dataset_phishing.csv')

print("Original Dataset Shape:", df.shape)

# 3. Handle Missing Values

df = df.dropna()

# 4. Encode Target Column

le = LabelEncoder()
df['status'] = le.fit_transform(df['status'])


# 5. Remove Non-Numeric Columns

if 'url' in df.columns:
    df = df.drop(['url'], axis=1)


# 6. Remove Outliers (IQR Method)

Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1

df_smooth = df[~((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))).any(axis=1)]

print("After Outlier Removal:", df_smooth.shape)


# 7. Feature Scaling

X = df_smooth.drop('status', axis=1)
y = df_smooth['status']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
OA = round(accuracy_score(y_test, y_pred) * 100, 2)
P = round(precision_score(y_test, y_pred) * 100, 2)
R = round(recall_score(y_test, y_pred) * 100, 2)
F1 = round(f1_score(y_test, y_pred) * 100, 2)


#feature selection

 #Remove highly correlated features
corr = X_train.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

X_train_reduced = X_train.drop(columns=to_drop)
X_test_reduced = X_test.drop(columns=to_drop)

# Select best features (adjust k as needed)
selector = SelectKBest(score_func=mutual_info_classif, k=20)

X_train_selected = selector.fit_transform(X_train_reduced, y_train)
X_test_selected = selector.transform(X_test_reduced)

selected_features = X_train_reduced.columns[selector.get_support()]

print("Selected features:", list(selected_features))

print("Overall Accuracy (OA):", OA, "%")
print("Precision (P):", P, "%")
print("Recall (R):", R, "%")
print("F1 Score:", F1, "%")

# convert back to dataframe
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# combine again
smooth_dataset = pd.concat([X_scaled, y.reset_index(drop=True)], axis=1)


# 8. Save Smoothed Dataset

smooth_dataset.to_csv("smoothed_dataset.csv", index=False)

print("Smoothed Dataset Shape:", smooth_dataset.shape)
print("Smoothed dataset saved as smoothed_dataset.csv")

Saving dataset_phishing.csv to dataset_phishing (3).csv
Original Dataset Shape: (11430, 89)
After Outlier Removal: (338, 88)


NameError: name 'accuracy_score' is not defined

In [ ]:
# 1. Import Libraries
import pandas as pd
import numpy as np
from google.colab import files

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, mutual_info_classif

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier


# 2. Upload Dataset
uploaded = files.upload()

df = pd.read_csv('dataset_phishing.csv')
print("Original Dataset Shape:", df.shape)


# 3. Handle Missing Values
df = df.dropna()


# 4. Encode Target Column
le = LabelEncoder()
df['status'] = le.fit_transform(df['status'])


# 5. Remove Non-Numeric Columns
if 'url' in df.columns:
    df = df.drop(['url'], axis=1)


# 6. Remove Outliers (IQR Method)
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1

df = df[~((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))).any(axis=1)]

print("After Outlier Removal:", df.shape)


# 7. Split Features & Target
X = df.drop('status', axis=1)
y = df['status']


# 8. Train/Test Split (MISSING IN YOUR CODE)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# 9. Feature Scaling
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns)


# =========================
# 🔥 FEATURE SELECTION FIXED
# =========================

# 10. Remove Highly Correlated Features
corr = X_train.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

X_train = X_train.drop(columns=to_drop)
X_test = X_test.drop(columns=to_drop)

print(f"Dropped {len(to_drop)} highly correlated features")


# 11. Select Best Features (Mutual Information)
selector = SelectKBest(score_func=mutual_info_classif, k=20)

X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

selected_features = X_train.columns[selector.get_support()]
print("Selected features:", list(selected_features))


# 12. Train Model (MISSING BEFORE METRICS)
model = RandomForestClassifier(random_state=42)
model.fit(X_train_selected, y_train)

y_pred = model.predict(X_test_selected)


# 13. Metrics (FIXED - NOW y_pred EXISTS)
OA = round(accuracy_score(y_test, y_pred) * 100, 2)
P = round(precision_score(y_test, y_pred) * 100, 2)
R = round(recall_score(y_test, y_pred) * 100, 2)
F1 = round(f1_score(y_test, y_pred) * 100, 2)

print("\nPerformance:")
print("Overall Accuracy (OA):", OA, "%")
print("Precision (P):", P, "%")
print("Recall (R):", R, "%")
print("F1 Score:", F1, "%")


# 14. Save Final Dataset (FIXED ALIGNMENT)
X_final = pd.DataFrame(X_train_selected, columns=selected_features)

smooth_dataset = pd.concat(
    [X_final.reset_index(drop=True),
     y_train.reset_index(drop=True)],
    axis=1
)

smooth_dataset.to_csv("smoothed_dataset.csv", index=False)

print("\nSmoothed Dataset Shape:", smooth_dataset.shape)
print("Saved as smoothed_dataset.csv")

Saving dataset_phishing.csv to dataset_phishing.csv
Original Dataset Shape: (11430, 89)
After Outlier Removal: (338, 88)
Dropped 3 highly correlated features
Selected features: ['nb_slash', 'nb_star', 'nb_dollar', 'nb_www', 'length_words_raw', 'char_repeat', 'shortest_word_path', 'longest_word_path', 'avg_words_raw', 'nb_hyperlinks', 'ratio_intHyperlinks', 'ratio_extHyperlinks', 'nb_extCSS', 'links_in_tags', 'safe_anchor', 'right_clic', 'domain_age', 'web_traffic', 'google_index', 'page_rank']

Performance:
Overall Accuracy (OA): 94.12 %
Precision (P): 95.65 %
Recall (R): 88.0 %
F1 Score: 91.67 %

Smoothed Dataset Shape: (270, 21)
Saved as smoothed_dataset.csv
